<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/10-partitioning-shuffle/03-shuffle-partitions-config.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 10.3 — spark.sql.shuffle.partitions: catch 200 red-handed

Watch the default fan a 41-row groupBy into 200 tasks, time the
overhead, do the sizing math from real shuffle bytes, then let AQE
right-size each exchange and prove it only merges — never splits.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("10.3-shuffle-partitions")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "false")   # meet the raw default first
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

SLOTS = spark.sparkContext.defaultParallelism
print("the untouched default:",
      spark.conf.get("spark.sql.shuffle.partitions"))

## 200 tasks for 41 rows

Run it, then open the Jobs tab: the second stage has 200 tasks,
~196 of which processed zero rows. Every one of them was scheduled,
launched, and reported.

In [ ]:
orders = spark.read.csv(f"{DATA_DIR}/orders.csv",
                        header=True, inferSchema=True)

by_country = orders.groupBy("country").count()
print("post-shuffle partitions:", by_country.rdd.getNumPartitions())
by_country.show()

## The overhead, on a stopwatch

Same medium-sized aggregation, three settings. On a laptop the
winner is near your slot count; 2000 pays pure scheduling tax.

In [ ]:
import time

big = (spark.range(10_000_000)
       .withColumn("k", (col("id") % 5000).cast("int"))
       .withColumn("payload", F.sha2(col("id").cast("string"), 256)))

for n in [str(2 * SLOTS), "200", "2000"]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    start = time.perf_counter()
    (big.groupBy("k").agg(F.count("*"), F.max("payload"))
        .write.format("noop").mode("overwrite").save())
    print(f"shuffle.partitions = {n:>4}: {time.perf_counter() - start:5.1f} s")

## The sizing math, from the real bill

Read the actual shuffle bytes off the REST API (10.2's trick),
then compute what the config SHOULD be for a 150 MB target. Toy
numbers here — but this is exactly the calculation for a 2 TB job.

In [ ]:
import urllib.request, json

app_id = spark.sparkContext.applicationId
stages = json.load(urllib.request.urlopen(
    f"http://localhost:4040/api/v1/applications/{app_id}/stages?status=complete"))

shuffle_bytes = max(s["shuffleWriteBytes"] for s in stages)
target = 150 * 1024 * 1024
ideal = max(1, round(shuffle_bytes / target))
ideal_rounded = max(SLOTS, -(-ideal // SLOTS) * SLOTS)   # ≥ slots, multiple of slots

print(f"biggest shuffle in this app: {shuffle_bytes:,} bytes")
print(f"÷ 150 MB target             = {ideal} partition(s)")
print(f"rounded to slot multiple    = {ideal_rounded}")
# for our megabytes the answer is 'your core count' — the right local setting

## AQE: plan 200, run right-sized

Same query, AQE on. The plan still says 200, but `AQEShuffleRead
coalesced` merges the actual blocks to the advisory size — check
the final partition count, then shrink the advisory size and watch
the count grow.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

agg = big.groupBy("k").agg(F.count("*"), F.max("payload"))

for advisory in ["64m", "4m", "1m"]:
    spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", advisory)
    print(f"advisory {advisory:>4} → {agg.rdd.getNumPartitions():>3} "
          f"partitions (planned: 200)")

agg.explain()   # find AQEShuffleRead coalesced

## The asymmetry: AQE never adds parallelism

Plan the same shuffle with a ceiling of 2: AQE leaves it at 2.
Merging down is possible; splitting up is not (only the skew-join
path splits, and only for joins — 7.4). Hence: generous ceiling.

In [ ]:
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")

spark.conf.set("spark.sql.shuffle.partitions", "2")
print("ceiling 2   →", agg.rdd.getNumPartitions(), "partitions — stuck at 2")

spark.conf.set("spark.sql.shuffle.partitions", "2000")
print("ceiling 2000 →", agg.rdd.getNumPartitions(),
      "partitions — AQE merged down to fit the data")

## Exercises

1. With AQE off, write `by_country` to parquet and count the output
   files at `shuffle.partitions` 200 vs 8. Reconcile the file count
   with the non-empty-partition count (5.4 callback).
2. Rerun the stopwatch cell with `spark.range(200_000)` (tiny) and
   `spark.range(50_000_000)` (bigger). Where does the crossover
   move? Explain using the waves model from 10.0.
3. Use the REST API to find the median task's shuffle read size in
   the 2000-partition run. How far is it from the 150 MB target?
   What real-world config mistake does this simulate?
4. Set `spark.sql.adaptive.coalescePartitions.initialPartitionNum`
   to 500 while `shuffle.partitions` stays 8, AQE on. Which number
   wins for the exchange, and what's the use case for having both?
5. Window functions shuffle too (Module 8). With AQE off, show that
   `Window.partitionBy("country")` over orders produces
   `shuffle.partitions` partitions, then reason about why AQE
   coalescing is just as effective there as for groupBy.

In [ ]:
# your exercise code here